# 学習済みbenchmark Baselineとの比較

別で学習したS2UTをBaselineとして登録し、その保存設定・preparedデータ・辞書を再利用してalignmentモデルを新規学習します。Baselineの重みを転移モデルへロードしません。
保存snapshotを別ディレクトリへ検証復元し、Baselineと転移モデルを同じnative fairseq生成器・vocoder・評価器で比較します。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import json, os, subprocess, sys, time
from pathlib import Path
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'
REPO_REVISION = 'PIN_AFTER_IMPLEMENTATION'
BENCHMARK = '/content/drive/MyDrive/ja-en-direct-s2st-benchmark-data/training/experiment-001' #@param {type:"string"}
RUN_NAME = 's2ut-training-experiment-001-paper-fisher-v1' #@param {type:"string"}
ROOT = '/content/drive/MyDrive/alignment-transfer-s2ut-data/benchmark-match-001' #@param {type:"string"}
RUN_TRANSFER = False #@param {type:"boolean"}
GENERATE_WAVEFORMS = False #@param {type:"boolean"}
ASR_BLEU = False #@param {type:"boolean"}
T2TT_UPDATES = 20000 #@param {type:"integer"}
ANNEAL_STEPS = 200000 #@param {type:"integer"}
PYTHON = '/content/alignment-venv/bin/python'
REPO = Path('/content/alignment-transfer-s2ut')
LOGS = Path(ROOT) / 'execution'
PREPARED = Path(BENCHMARK) / 'data/s2ut/fairseq'
BACKUP = Path(BENCHMARK) / 'checkpoints' / RUN_NAME
RESTORED = Path(ROOT) / 'baseline-snapshot'
BASELINE = Path(ROOT) / 'baseline.json'
MANIFEST = Path(ROOT) / 'manifest.jsonl'
os.environ['HF_HOME'] = '/content/huggingface'
os.environ['PYTHONUTF8'] = '1'
os.environ['MPLCONFIGDIR'] = '/content/matplotlib-cache'
assert not Path(ROOT).resolve().is_relative_to(Path(BENCHMARK).resolve())
assert not ASR_BLEU or GENERATE_WAVEFORMS


## 固定環境の構築

Python 3.10 / PyTorch 2.7.1 / 公式fairseq固定revisionを構築します。


In [ ]:
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', REPO_REVISION], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', REPO_REVISION], check=True)
os.chdir(REPO)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'uv'], check=True)
subprocess.run(['uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv'], check=True)
for packages in [
    ['pip==23.3.2', 'setuptools<81', 'wheel'],
    ['torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4', 'Cython==3.2.9'],
    ['--no-build-isolation', '-r', 'requirements-fairseq.txt'],
    ['-r', 'requirements.txt'],
]:
    subprocess.run([PYTHON, '-m', 'pip', 'install', *packages], check=True)
subprocess.run([PYTHON, '-m', 'pip', 'check'], check=True)

def run(name, module, *arguments, gpu=False, env=None):
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', name, '--logs', str(LOGS)]
    if gpu:
        command.append('--monitor-gpu')
    command += ['--', PYTHON, '-m', module, *map(str, arguments)]
    return subprocess.run(command, check=True, env=env)

run('runtime', 'alignment_transfer_s2ut.pilot', 'runtime')
runtime = json.loads(subprocess.check_output([PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'runtime'], text=True))
(Path(ROOT) / 'runtime.json').write_text(json.dumps(runtime, indent=2), encoding='utf-8')
print(runtime)
run('cpu-tests', 'pytest', '-q', '-m', 'not gpu')


## native FP16接続試験

小規模な人工データで公式TrainerのFP16更新とpriorのFP32保持を確認します。GPUランタイムで実行してください。実CTC/HuBERT/vocoderを含む統合試験とは別です。


In [ ]:
subprocess.run([PYTHON, '-m', 'pytest', 'tests/test_benchmark_native.py::test_native_fp16_trainer_and_fp32_prior',
    '-q', '--basetemp', '/content/native-fp16-test'], cwd=REPO, check=True)


## 既存Baselineの復元と登録

Driveで確認した保存形式は`snapshot.json`と`snapshot.zip`です。初回は最新snapshotを選び、以後の再開では同じsnapshotを維持します。新しいBaselineで比較するときは新しいROOTを指定します。元のbackup/preparedデータは変更しません。


In [ ]:
run('restore-baseline', 'alignment_transfer_s2ut.benchmark', 'restore', '--backup', BACKUP, '--output', RESTORED, '--resume')
run('register-baseline', 'alignment_transfer_s2ut.benchmark', 'register',
    '--checkpoint', RESTORED / 'checkpoints/checkpoint_best.pt', '--data', PREPARED,
    '--recipe-lock', RESTORED / 's2ut-recipe-lock.json', '--metadata', RESTORED / 'research-metadata.json',
    '--output', BASELINE, '--resume')
baseline = json.loads(BASELINE.read_text())
print({key: baseline[key] for key in ['checkpoint_updates', 'training_budget', 'checkpoint_selection', 'selection_policy']})
assert 1 <= ANNEAL_STEPS < baseline['training_budget']

BACKBONE_CONFIG = Path(ROOT) / 'backbone.json'
BACKBONE_CONFIG.write_text(json.dumps({'backbone': baseline['backbone']}), encoding='utf-8')


## 同一のtrain/devとunitを使用

少量pilotへ抽出せず、Baselineが使った全train/devを読み込みます。testはteacher作成・モデル選択に使いません。HuBERT unitはbenchmarkの保存済み列をimportし、prepared targetと一致することを確認します。


In [ ]:
run('full-manifest', 'alignment_transfer_s2ut.benchmark_data', 'manifest',
    '--common', Path(BENCHMARK) / 'data/common', '--prepared', PREPARED, '--output', MANIFEST, '--resume')
run('import-units', 'alignment_transfer_s2ut.benchmark_data', 'units', '--manifest', MANIFEST,
    '--benchmark-units', Path(BENCHMARK) / 'data/s2ut/units', '--prepared', PREPARED,
    '--output', Path(ROOT) / 'units', '--resume')
rows = [json.loads(line) for line in MANIFEST.read_text().splitlines() if line.strip()]


## T2TT学習・BLEU・alignment準備

翻訳teacherは同じ全train pairから学習し、dev text BLEUとalignment診断を確認します。T2TTの学習はS2UT Baselineの継続学習とは別です。


In [ ]:
run('t2tt', 'alignment_transfer_s2ut.train_t2tt', '--manifest', MANIFEST,
    '--config', BACKBONE_CONFIG, '--output', Path(ROOT) / 't2tt', '--max-updates', T2TT_UPDATES,
    '--device', 'cuda', '--loader-workers', 2, '--resume-if-exists', gpu=True)
run('t2tt-dev', 'alignment_transfer_s2ut.evaluate_t2tt', '--manifest', MANIFEST,
    '--checkpoint', Path(ROOT) / 't2tt/checkpoint_best.pt', '--output', Path(ROOT) / 't2tt/dev-text.json',
    '--device', 'cuda', '--resume', gpu=True)
print(json.loads((Path(ROOT) / 't2tt/dev-text.json').read_text()))
run('attention', 'alignment_transfer_s2ut.extract_t2tt_attention', '--manifest', MANIFEST,
    '--checkpoint', Path(ROOT) / 't2tt/checkpoint_best.pt', '--output', Path(ROOT) / 'mt', '--device', 'cuda', '--resume', gpu=True)
run('ctc', 'alignment_transfer_s2ut.align_speech_text', '--manifest', MANIFEST,
    '--config', 'configs/alignment/ctc.yaml', '--output', Path(ROOT) / 'ctc', '--device', 'cuda', '--resume', gpu=True)
run('teacher', 'alignment_transfer_s2ut.build_teacher_alignment', '--manifest', MANIFEST,
    '--mt-attention', Path(ROOT) / 'mt', '--alignment', Path(ROOT) / 'ctc', '--units', Path(ROOT) / 'units',
    '--config', BACKBONE_CONFIG, '--output', Path(ROOT) / 'teacher', '--resume')
run('diagnostics', 'alignment_transfer_s2ut.diagnose_alignment', '--manifest', MANIFEST,
    '--teacher', Path(ROOT) / 'teacher', '--units', Path(ROOT) / 'units', '--config', BACKBONE_CONFIG,
    '--split', 'dev', '--max-samples', 500, '--heatmaps', 20, '--output', Path(ROOT) / 'audit', '--overwrite')


## 保存済みBaselineと同じ条件で転移モデルを新規学習

保存済みcfgから公式fairseq Trainer・FP16・token-budget batch・optimizer・scheduler・補助lossを構築します。更新数はBaselineの完了済み学習budgetに合わせます。report-only unit-loss選択runでは同じ間隔のdev unit lossでbestを選択します。planは学習を開始しません。


In [ ]:
TRANSFER = Path(ROOT) / 'annealed-t2tt'
TRANSFER_ARGS = ['--baseline', str(BASELINE), '--output', str(TRANSFER), '--method', 'annealed_guidance',
    '--manifest', str(MANIFEST), '--units', str(Path(ROOT) / 'units'), '--teacher', str(Path(ROOT) / 'teacher'),
    '--anneal-steps', str(ANNEAL_STEPS), '--resume']
plan = json.loads(subprocess.check_output([PYTHON, '-m', 'alignment_transfer_s2ut.benchmark', 'plan', *TRANSFER_ARGS], text=True))
print({'training_budget': plan['training_budget'], 'initialization': plan['initialization'], 'selection': plan['selection_policy']})
if RUN_TRANSFER:
    run('native-transfer', 'alignment_transfer_s2ut.benchmark', 'train', *TRANSFER_ARGS, gpu=True)


## 共通のteacher-free推論とdev評価

Baselineを再学習せず、両モデルで同じnative fairseq beam 10 / max_len_a 1 / max_len_b 200を使います。転移モデルも推論時はstock S2UTとして読み込み、teacherを必要としません。ASR-BLEUは両モデル共通の設定です。


In [ ]:
TRANSFER_RECORD = Path(ROOT) / 'annealed-t2tt.json'
if (TRANSFER / 'checkpoints/checkpoint_best.pt').is_file():
    run('register-transfer', 'alignment_transfer_s2ut.benchmark', 'register-transfer', '--run', TRANSFER,
        '--checkpoint', TRANSFER / 'checkpoints/checkpoint_best.pt', '--output', TRANSFER_RECORD, '--resume')
    records = [BASELINE, TRANSFER_RECORD]
    run('compare-training', 'alignment_transfer_s2ut.benchmark', 'compare', *records, '--output', Path(ROOT) / 'training-comparison.json', '--resume')
    source_only = Path(ROOT) / 'dev-source.jsonl'
    run('dev-source', 'alignment_transfer_s2ut.prepare_inference_input', '--manifest', MANIFEST, '--split', 'dev', '--output', source_only, '--resume')
    if GENERATE_WAVEFORMS:
        run('vocoder', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/s2ut/vocoder.yaml')
    evaluations = []
    for name, record in zip(['baseline', 'annealed-t2tt'], records):
        predictions = Path(ROOT) / 'evaluation' / name / 'dev-predictions.jsonl'
        vocoder = ['--vocoder-config', 'configs/s2ut/vocoder.yaml'] if GENERATE_WAVEFORMS else ['--units-only']
        run(name + '-infer', 'alignment_transfer_s2ut.benchmark', 'infer', '--record', record, '--input', source_only,
            '--output', predictions, '--device', 'cuda', '--resume', *vocoder, gpu=True)
        metrics = predictions.with_name('dev-metrics.json')
        asr = ['--asr-config', 'configs/experiments/evaluation.yaml', '--device', 'cuda'] if ASR_BLEU else []
        run(name + '-evaluate', 'alignment_transfer_s2ut.evaluate', '--manifest', MANIFEST, '--split', 'dev',
            '--units', Path(ROOT) / 'units', '--predictions', predictions, '--output', metrics, '--resume', *asr, gpu=ASR_BLEU)
        evaluations.append(metrics)
    run('comparison', 'alignment_transfer_s2ut.benchmark', 'compare', *records, '--evaluations', *evaluations,
        '--output', Path(ROOT) / 'comparison.json', '--resume')
    print(json.loads((Path(ROOT) / 'comparison.json').read_text()))
else:
    print('転移モデルの学習完了後にこのセルを実行してください。Baselineは登録済みです。')
